# E6 — Ablation độ phân giải: nguồn C 416 gốc vs 640 upscale (YOLO26s, 15 epochs/arm)

**Câu hỏi:** bước chuẩn hóa 640×640 (letterbox + Lanczos upscale 11.110 ảnh 416px của nguồn C, mục 3.4 báo cáo)
có làm MẤT thông tin ảnh hưởng đến kết quả không — đặc biệt các lớp vật thể nhỏ (goggles ~0,34% diện tích ảnh, gloves, mask)?

**Thiết kế (2 cánh tay, cô lập đúng một biến):**
- Chỉ dùng **subset ảnh nguồn C** (lọc qua `manifest.csv`, giữ nguyên split group-aware — không re-split, không leakage).
- **Arm `res640`**: ảnh C bản 640 upscale như trong PPE-Unified v1, train/val/test imgsz=640.
- **Arm `res416`**: ảnh C bản GỐC 416 (Add Input dataset backup `ppe-unified-v1-416c`); train/val/test imgsz=416.
  Nếu không có backup, notebook fallback: downscale bản 640 về 416 (Lanczos) — *ghi rõ caveat trong báo cáo: không phải ảnh gốc.*
- Cùng YOLO26s, seed 42, lr0 0,01, optimizer auto, batch 16, 15 epochs, RFS tắt (tránh nhiễu chéo với E5).

**Đầu ra:** `results/e6_resolution.csv` — mAP@50/mAP@50-95 tổng + AP@50 các lớp nhỏ (goggles, no_goggles, gloves, no_gloves, mask)
cho từng arm + chênh lệch. Diễn giải: nếu 640-upscale ≥ 416-gốc thì việc chuẩn hóa không gây hại (vì model nhận thêm ngữ cảnh pixel);
nếu 416-gốc thắng ở lớp nhỏ thì đó là chi phí thật của chuẩn hóa — cần báo cáo trong Hạn chế.

**Add Input:** `ppe-unified-v1` (+ `ppe-unified-v1-416c` nếu có). GPU T4 x2 · Internet ON. Ước tính ~2,5h. Có checkpoint-resume.

In [ ]:
KEY_PREFIX = "e6_"
# ==================== CONFIG ====================
QUICK_TEST = False             # True = chạy thử 1 epoch/vài batch để kiểm tra pipeline; số liệu quick KHÔNG đưa vào báo cáo
SEED = 42

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: tự nạp state.json từ output phiên trước của CHÍNH notebook này (nếu Add Input)
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) for k in _s): STATE.update(_s); print("Resumed state:", list(_s))
def save_state():
    json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Bật GPU T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")


In [ ]:
from pathlib import Path
import yaml, pandas as pd

def find_dataset_root():
    """Tìm PPE_Unified_v1 trong /kaggle/input (glob đệ quy — không khóa path)."""
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Chưa Add Input dataset ppe-unified-v1!")

def find_manifest():
    """manifest.csv truy vết nguồn gốc (source A/B/C) của từng ảnh — sinh bởi build_unified_dataset.py."""
    for c in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
        return pd.read_csv(c)
    raise AssertionError("Không thấy manifest.csv trong dataset input!")

DS_ROOT = find_dataset_root()
MANIFEST = find_manifest()
print("Dataset:", DS_ROOT, "| manifest:", len(MANIFEST), "dòng")
print(MANIFEST["source"].value_counts() if "source" in MANIFEST.columns else MANIFEST.columns.tolist())


In [ ]:
# ---------- Dựng 2 dataset arm chỉ gồm ảnh nguồn C (giữ nguyên split) ----------
from PIL import Image

def build_subset_C(dst_name, resolution):
    """Copy các ảnh source C theo split gốc; resolution 640 => copy nguyên bản từ PPE-Unified;
    resolution 416 => ưu tiên bản gốc 416 backup, fallback downscale Lanczos."""
    dst = f"{WORK}/{dst_name}"
    if os.path.isdir(dst): shutil.rmtree(dst)
    # tìm backup 416 (nếu người dùng Add Input)
    ROOT416 = None
    for c in glob.glob("/kaggle/input/**/416*/**/images", recursive=True):
        ROOT416 = os.path.dirname(c); break
    m = MANIFEST[MANIFEST["source"].str.upper().str.startswith("C")]
    n_fallback = 0
    for _, row in m.iterrows():
        split, fname = row["split"], row["file"]
        for sub in ("images", "labels"):
            os.makedirs(f"{dst}/{split}/{sub}", exist_ok=True)
        lbl = fname.rsplit(".", 1)[0] + ".txt"
        shutil.copy(f"{DS_ROOT}/{split}/labels/{lbl}", f"{dst}/{split}/labels/{lbl}")
        src_img = f"{DS_ROOT}/{split}/images/{fname}"
        if resolution == 640:
            shutil.copy(src_img, f"{dst}/{split}/images/{fname}")
        else:  # 416
            orig = glob.glob(f"{ROOT416}/**/{fname}", recursive=True) if ROOT416 else []
            if orig:
                shutil.copy(orig[0], f"{dst}/{split}/images/{fname}")
            else:  # FALLBACK: downscale — không phải ảnh gốc, chỉ xấp xỉ
                Image.open(src_img).resize((416, 416), Image.LANCZOS).save(f"{dst}/{split}/images/{fname}")
                n_fallback += 1
    names = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    dyaml = f"{dst}/data.yaml"
    yaml.dump({"path": dst, "train": "train/images", "val": "val/images", "test": "test/images",
               "nc": len(names), "names": names}, open(dyaml, "w"))
    if n_fallback: print(f"CAVEAT: {n_fallback} ảnh dùng fallback downscale (không có backup 416 gốc)")
    return dyaml


In [ ]:
# ---------- Train + eval 2 arm ----------
from ultralytics import YOLO
import pandas as pd
SMALL = ["goggles", "no_goggles", "gloves", "no_gloves", "mask"]
EPOCHS = 1 if QUICK_TEST else 15
rows = []
for arm, res in [("res640", 640), ("res416", 416)]:
    key = f"e6_{arm}"
    if done(key): rows.append(STATE[key]); print("skip:", key); continue
    dyaml = build_subset_C(f"dsC_{arm}", res)
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    m.train(data=dyaml, epochs=EPOCHS, imgsz=res, batch=16, seed=SEED, lr0=0.01,
            project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
    t = m.val(data=dyaml, split="test", imgsz=res, batch=16)
    names = list(t.names.values())
    per = {f"AP50_{n}": round(float(a), 4) for n, a in zip(names, t.box.ap50)}
    rec = {"arm": arm, "imgsz": res, "mAP50_test": round(float(t.box.map50), 4),
           "mAP50_95_test": round(float(t.box.map), 4),
           **{k: per.get(k) for k in [f"AP50_{c}" for c in SMALL]},
           "min": round((time.time() - t0) / 60, 1)}
    STATE[key] = rec; save_state(); rows.append(rec)
df = pd.DataFrame(rows); df.to_csv(f"{WORK}/results/e6_resolution.csv", index=False)
print(df.to_string(index=False))


In [ ]:
# ==================== DỌN OUTPUT: KHÔNG kèm dataset ====================
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Output cuối:", sorted(os.listdir(WORK)))
